# Autenticação de Saída (Outbound Auth)

Outbound Auth permite que agentes e o AgentCore Gateway acessem com segurança recursos AWS e serviços de terceiros em nome de usuários que foram autenticados e autorizados durante Inbound Auth. Para integrar autorização com um recurso AWS ou serviço de terceiros, é necessário configurar tanto Inbound Auth quanto Outbound Auth.

Com acesso suficiente e delegação de permissão segura suportada pelo AgentCore Identity, agentes podem acessar perfeitamente e com segurança recursos AWS e ferramentas de terceiros como GitHub, Google, Salesforce e Slack. Agentes podem realizar ações nesses serviços em nome de usuários ou independentemente, desde que haja consentimento pré-autorizado do usuário. Adicionalmente, você pode reduzir fadiga de consentimento usando um cofre de tokens seguro e criar experiências simplificadas de agentes de IA.

## Configuração de Autenticação de Saída

Primeiro, você registra sua aplicação cliente com provedores de terceiros e então cria um Outbound Auth. Você especifica como deseja validar acesso ao recurso AWS ou serviço de terceiros ou alvos AgentCore Gateway. Você pode usar OAuth 2LO/3LO ou chaves API. Com OAuth, você pode selecionar de provedores que AgentCore Identity fornece. Nesse caso, você insere os detalhes de configuração para os provedores do AgentCore Identity. Alternativamente, você pode fornecer detalhes para um provedor personalizado.

Quando um usuário quer acesso a um recurso AWS ou serviço de terceiros ou alvo AgentCore Gateway, o Outbound Auth confirma que os tokens de acesso fornecidos pelo Incoming Auth são válidos e, se for o caso, permite acesso ao recurso.

<div style="text-align:center">
    <img src="images/outbound_auth.png" width="90%"/>
</div>

## Provedores de credenciais de recursos

Este é um componente que o código do agente usa para recuperar credenciais de servidores de recursos downstream (ex.: Google, GitHub) para acessá-los, ex.: buscar emails do Gmail, adicionar uma reunião ao Google Calendar. Ele remove o trabalho pesado dos desenvolvedores de agentes implementando fluxos de orquestração OAuth2 2LO e 3LO entre usuários finais, código do agente e servidores de autorização externos. AgentCore fornece tanto um provedor de credenciais OAuth2 personalizado quanto uma lista de provedores integrados como Google, GitHub, Slack, Salesforce com endpoint do servidor de autorização e parâmetros específicos do provedor pré-preenchidos.

Bedrock AgentCore Identity fornece OAuth2 e Provedores de Credenciais de Chave API para desenvolvedores de agentes autenticarem com recursos externos que suportam OAuth2 ou chave API. No exemplo a seguir, vamos guiá-lo através da configuração de um provedor de credenciais de Chave API. Um agente pode então usar o provedor de credenciais de Chave API para recuperar a chave API para quaisquer operações do agente. Por favor, consulte a documentação para os outros provedores de credenciais.

### Criando um provedor de credenciais de recurso.

Aqui está um exemplo de criação de um provedor de credenciais de recurso de Chave API.

```
from bedrock_agentcore.services.identity import IdentityClient
identity_client = IdentityClient(region="us-west-2")

api_key_provider = identity_client.create_api_key_credential_provider({
    "name": "APIKey-provider-name",
    "apiKey": "<my-api-key>" # Substitua pela chave API que você obtém do fornecedor de aplicação externa, ex.: OpenAI
})
print(api_key_provider)
```

### Recuperando tokens de acesso ou chaves API do provedor de credenciais de Recurso.

Aqui está um exemplo de recuperação da chave API do provedor de credenciais de Chave API. O agente pode usar a chave API para interagir com serviços como um LLM ou outros serviços que usam uma configuração de chave API. Para recuperar credenciais como access_token ou chave API do provedor de credenciais, você pode decorar sua função conforme mostrado abaixo.

```
import asyncio
from bedrock_agentcore.identity.auth import requires_access_token, requires_api_key

@requires_api_key(
    provider_name="APIKey-provider" # substitua pelo nome do seu próprio provedor de credenciais
)
async def need_api_key(*, api_key: str):
    print(f'received api key for async func: {api_key}')

await need_api_key(api_key="")
```

Aqui estão os vários parâmetros que você pode usar com o decorador @require_access_token.


| Nome do Parâmetro    | Descrição                                                                |
|:---------------------|:-------------------------------------------------------------------------|
| provider_name        | O nome do provedor de credenciais                                        |
| into                 | Nome do parâmetro para injetar o token                                   |
| scopes               | Escopos OAuth2 a serem solicitados                                       |
| on_auth_url	       | Callback para manipular URLs de autorização                              |
| auth_flow            | Tipo de fluxo de autenticação ("M2M" ou "USER_FEDERATION")               |
| callback_url         | URL de callback OAuth2                                                   |
| force_authentication | Forçar re-autenticação                                                   |
| token_poller         | Implementação personalizada do token poller                              |

		


# Hospedando Strands Agents com modelos OpenAI no Amazon Bedrock AgentCore Runtime

## Visão Geral


Neste tutorial, modificaremos o agente que você implantou em 01-AgentCore-runtime, que usa modelo openai e o configuraremos para Outbound Auth usando o provedor de credenciais de Chave API. Você configurará um provedor de credenciais de Chave API para armazenar a chave open-ai e modificará o código do agente para usar esta chave.

### Arquitetura do Tutorial

<div style="text-align:center">
    <img src="images/outbound_auth_api.png" width="90%"/>
</div>


### Detalhes do Tutorial

| Informação          | Detalhes                                                                      |
|:--------------------|:------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                |
| Tipo de agente      | Único                                                                         |
| Framework Agêntico  | Strands Agents                                                                |
| Modelo LLM          | GPT 4.1 mini                                                                  |
| Componentes         | Hospedagem de agente no AgentCore Runtime. Usando Strands Agent e OpenAI Model |
| Vertical            | Cross-vertical                                                                |
| Complexidade        | Fácil                                                                         |
| SDK usado           | Amazon BedrockAgentCore Python SDK e boto3                                    |
| Provedor Credential | Tipo : Chave API                                                              |


### Funcionalidades Chave do Tutorial

* Hospedagem de Agentes no Amazon Bedrock AgentCore Runtime
* Uso de modelos OpenAI
* Uso de Strands Agents
* Uso de AgentCore egress Auth com provedor de credenciais de Chave API.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker em execução
* Chaves API OpenAI

Como obter chaves API OpenAI:
- OpenAI: [Chaves API OpenAI](https://help.openai.com/en/articles/4936850-where-do-i-find-my-openai-api-key)
- Azure OpenAI: [Criar recurso Azure OpenAI e obter chaves](https://learn.microsoft.com/azure/ai-services/openai/how-to/create-resource?tabs=azure-portal)

Definir variáveis de ambiente antes de executar:
- OpenAI:
  - `OPENAI_API_KEY`
- Azure OpenAI:
  - `AZURE_OPENAI_API_KEY`
  - `AZURE_OPENAI_ENDPOINT`
  - `AZURE_OPENAI_API_VERSION` (ex.: `2024-02-15-preview`)
  - `AZURE_OPENAI_DEPLOYMENT` (nome do seu modelo implantado)

Opção de troca de provedor:
- `OPENAI_PROVIDER` = `openai` (padrão) ou `azure`

NOTA:
- Não codifique segredos diretamente. Use o provedor de credenciais do AgentCore Identity para armazenar e recuperar chaves API em tempo de execução, e rotacione chaves regularmente.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Criando seus agentes e experimentando localmente

Antes de implantar nossos agentes no AgentCore Runtime, vamos desenvolvê-los e executá-los localmente para fins de experimentação.

Para aplicações agênticas de produção, precisaremos desacoplar o processo de criação do agente do processo de invocação. Com AgentCore Runtime, decoraremos a parte de invocação do nosso agente com o decorador `@app.entrypoint` e o teremos como ponto de entrada para nosso runtime. Vamos primeiro ver como cada agente é desenvolvido durante a fase de experimentação.

A arquitetura aqui será a seguinte:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os

##Update the below configuration with your Azure API Key details.
os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_open_ai(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_open_ai(json.loads(args.payload))
    print(response)

#### Invocando agente local

In [ ]:
!python strands_agents_openai.py '{"prompt": "What is the weather now?"}'

## Criar um Provedor de Credenciais de Recurso

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient

from boto3.session import Session
import boto3

boto_session = Session()
region = boto_session.region_name

# Configure API Key Provider
identity_client = IdentityClient(region=region)

api_key_provider = identity_client.create_api_key_credential_provider(
    {
        "name": "openai-apikey-provider",
        "apiKey": "<YOUR_API_KEY>",  # Replace it with the API key you obtain from the external application vendor, e.g., OpenAI
    }
)
print(api_key_provider)

## Preparando seu agente para implantação no AgentCore Runtime e usar o Provedor de Credenciais de Recurso

Vamos agora implantar nossos agentes no AgentCore Runtime. Para fazer isso, precisamos:
* Importar o Runtime App com `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Inicializar o App no nosso código com `app = BedrockAgentCoreApp()`
* Decorar a função de invocação com o decorador `@app.entrypoint`
* Deixar AgentCoreRuntime controlar a execução do agente com `app.run()`
* Recuperar a chave openAI do provedor de credenciais de recurso criado na etapa anterior

### Strands Agents com modelo OpenAI
Vamos começar com nosso Strands Agent usando o modelo GPT 4.1 mini. Todos os outros funcionarão exatamente da mesma forma.

In [ ]:
%%writefile strands_agents_openai.py
import asyncio
from bedrock_agentcore.identity.auth import requires_access_token, requires_api_key
from strands import Agent, tool
from strands_tools import calculator 
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp

AZURE_API_KEY_FROM_CREDS_PROVIDER = ""


@requires_api_key(
    provider_name="openai-apikey-provider" # replace with your own credential provider name
)
async def need_api_key(*, api_key: str):
    global AZURE_API_KEY_FROM_CREDS_PROVIDER
    print(f'received api key for async func: {api_key}')
    AZURE_API_KEY_FROM_CREDS_PROVIDER = api_key

# Don't print empty value at module level - will print in entrypoint function

app = BedrockAgentCoreApp()

# API key will be set dynamically in the entrypoint function
#Update the below configuration with your Azure API Key details.
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

# Global agent variable
agent = None

@app.entrypoint
async def strands_agent_open_ai(payload):
    """
    Invoke the agent with a payload
    """
    global AZURE_API_KEY_FROM_CREDS_PROVIDER, agent
    
    print(f"Entrypoint called with AZURE_API_KEY_FROM_CREDS_PROVIDER: '{AZURE_API_KEY_FROM_CREDS_PROVIDER}'")
    
    # Get API key if not already retrieved
    if not AZURE_API_KEY_FROM_CREDS_PROVIDER:
        print("Attempting to retrieve API key...")
        try:
            await need_api_key(api_key="")
            print(f"API key retrieved: '{AZURE_API_KEY_FROM_CREDS_PROVIDER}'")
            os.environ["AZURE_API_KEY"] = AZURE_API_KEY_FROM_CREDS_PROVIDER
            print("Environment variable AZURE_API_KEY set")
        except Exception as e:
            print(f"Error retrieving API key: {e}")
            raise
    else:
        print("API key already available")
    
    # Initialize agent after API key is set
    if agent is None:
        print("Initializing agent with API key...")
        model = "azure/gpt-4.1-mini"
        litellm_model = LiteLLMModel(
            model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
        )
        
        agent = Agent(
            model=litellm_model,
            tools=[calculator, weather],
            system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
        )
        print("Agent initialized successfully")
    
    user_input = payload.get("prompt")
    print(f"User input: {user_input}")
    
    try:
        response = agent(user_input)
        print(f"Agent response: {response}")
        return response.message['content'][0]['text']
    except Exception as e:
        print(f"Error in agent processing: {e}")
        raise

if __name__ == "__main__":
    app.run()


## O que acontece por trás das cenas?

Quando você usa `BedrockAgentCoreApp`, ele automaticamente:

* Cria um servidor HTTP que escuta na porta 8080
* Implementa o endpoint `/invocations` necessário para processar os requisitos do agente
* Implementa o endpoint `/ping` para verificações de saúde (muito importante para agentes assíncronos)
* Manipula tipos de conteúdo apropriados e formatos de resposta
* Gerencia tratamento de erros de acordo com os padrões AWS

## Implantando o agente no AgentCore Runtime

A operação `CreateAgentRuntime` suporta opções abrangentes de configuração, permitindo que você especifique imagens de container, variáveis de ambiente e configurações de criptografia. Você também pode configurar definições de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente.

**Nota:** A melhor prática de operações é empacotar código como container e enviar para ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCore Python SDK para facilmente empacotar seus artefatos e implantá-los no AgentCore runtime.

### Configurar implantação AgentCore Runtime

Em seguida, usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR na inicialização.

Durante a etapa de configuração, seu docker file será gerado com base no código da sua aplicação

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import boto3
import json

boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()
agent_name = "strands_agents_openai"

response = agentcore_runtime.configure(
    entrypoint="strands_agents_openai.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    agent_name=agent_name,
    requirements_file="requirements.txt",
    region=region,
)
response

### Iniciando agente no AgentCore Runtime

Agora que temos um docker file, vamos iniciar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

#### Adicionar políticas extras necessárias à role auto-criada

Como estamos adicionando alguma identidade de saída ao nosso agente, precisaremos obter algumas Chaves API e Secrets que não estão disponíveis na role auto-criada. Para fazer isso, precisaremos adicionar algumas permissões extras à nossa role IAM auto-criada. Vamos primeiro obter esta role e então adicionar essas permissões a ela.

In [ ]:
import json

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

runtime_response = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
runtime_role = runtime_response["roleArn"]

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "GetResourceAPIKey",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:GetResourceApiKey"],
            "Resource": "*",
        },
        {
            "Sid": "SecretManager",
            "Effect": "Allow",
            "Action": ["secretsmanager:GetSecretValue"],
            "Resource": "arn:aws:secretsmanager:*:*:secret:bedrock-agentcore*",
        },
    ],
}
iam_client = boto3.client("iam", region_name=region)

response = iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)

### Verificando o Status do AgentCore Runtime
Agora que implantamos o AgentCore Runtime, vamos verificar seu status de implantação

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### Invocando AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "Hello"}, user_id="userid_1234567890"
)
invoke_response

### Processando resultados de invocação

Agora podemos processar nossos resultados de invocação para incluí-los em uma aplicação

In [ ]:
from IPython.display import Markdown, display

response_text = invoke_response["response"][0]
display(Markdown(response_text))

### Invocando AgentCore Runtime com boto3

Agora que seu AgentCore Runtime foi criado, você pode invocá-lo com qualquer AWS SDK. Por exemplo, você pode usar o método `invoke_agent_runtime` do boto3 para isso.

In [ ]:
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    runtimeUserId="userid_1234567890",
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How much is 2X2?"}),
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                logger.info(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Limpeza (Opcional)

Vamos agora limpar o AgentCore Runtime criado

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

iam_client = boto3.client("iam")

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split("/")[1], force=True
)

policies = iam_client.list_role_policies(
    RoleName=runtime_role.split("/")[1], MaxItems=100
)

for policy_name in policies["PolicyNames"]:
    iam_client.delete_role_policy(
        RoleName=runtime_role.split("/")[1], PolicyName=policy_name
    )
iam_response = iam_client.delete_role(RoleName=runtime_role.split("/")[1])

# Parabéns!